In [ ]:
import pandas as pd
import numpy as np

In [ ]:
cars = pd.read_csv("https://raw.githubusercontent.com/juliandnl/redi_ss20/master/cars.csv")

# Part 1 - Transformations

The following exercises are about doing row-wise transformations of string columns. [Docs](https://pandas.pydata.org/docs/user_guide/text.html#) on working with string data in pandas. [Built-in functions](https://pandas.pydata.org/docs/reference/series.html#string-handling) on string columns. Note that the exercises cover different implementation variations.

As a rule of thumb it is better to use dataframe built-in functionality if available for the logic you want to implement. Lower-level abstractions like `apply` can help if the functionality is not available.

## Exercise 1
Create a new column `has_power` that indicates (with booleans) if the value in the `Variant` column contains the word "Power"

1. Use `apply` with a `def` function and anonymous `lambda` function
2. Use only dataframe built-in functionality



In [ ]:
# 1. Use `apply`

# with def
def has_power(car_variant):
  return 'Power' in car_variant

cars['has_power'] = cars['Variant'].apply(has_power)

# with lambda
cars['has_power'] = cars['Variant'].apply(lambda car_variant: 'Power' in car_variant)

# 2. Use only dataframe built-in functionality
cars['has_power'] = cars['Variant'].str.contains('Power')

## Exercise 2
Create an abbreviation of the `Make` of a car. Create a new column `abbrev_brand` that contains the first 3 letters in uppercase of the value from the `Make` columns. For example, if `Make` is `Chrysler`, `abbrev_brand` should be `CHR`.

1. Use `apply` with a regular `def` function and anonymous `lambda` function
2. Use only dataframe built-in functionality

In [ ]:
# 1. Use `apply`

# with def
def create_abbrev(make):
  return make[0:3].upper()

cars.loc[:, 'abbrev_brand'] = cars.loc[:, 'Make'].apply(create_abbrev)

# with lambda
cars.loc[:, 'abbrev_brand'] = cars.loc[:, 'Make'].apply(lambda make: make[0:3].upper())

# 2. Use only dataframe built-in functionality
cars.loc[:, 'abbrev_brand'] = cars.loc[:, 'Make'].str[0:3].str.upper()

# Part 2 - Joins

The following exercises cover different join types. Check the [merge](https://pandas.pydata.org/docs/reference/api/pandas.merge.html) docs for help on the `how` parameter.

We create the same dataframe as seen in the class to use for the joins. It is computed based on 'cars' and has following properties
- Average price **in euros €** of car models per brand (brand = Make). Columns: `Make, Model, avg_price`
- 3 additional "invented" rows
- All `Mercedes Benz` cars are removed

In [ ]:
peso2euro_rate = 0.0045 # fluctuating value
cars.loc[:, 'price_euros'] = peso2euro_rate * cars.loc[:, 'Price']

avg_model_prices = cars.groupby(['Make', 'Model'])['price_euros'].mean().reset_index()
avg_model_prices = avg_model_prices.rename(columns={'price_euros': 'avg_price'})
avg_model_prices = avg_model_prices.loc[avg_model_prices.loc[:, 'Make'] != 'Mercedes Benz', :]

invented_rows = pd.DataFrame(
    data = [('Ford', 'Lo', 158703.340), ('Ford', 'Hi', 324235.670), ('Ford', 'Cheap', 6533.700)],
    columns=['Make', 'Model', 'avg_price']
)

avg_model_prices = pd.concat([invented_rows, avg_model_prices], axis=0, ignore_index=True)

avg_model_prices.shape

In [ ]:
filter_mercedes_rows = lambda df: df.loc[df.loc[:, 'Make'] == 'Mercedes Benz', :]
filter_invented_rows = lambda df: df.loc[df.loc[:, 'Model'].isin(['Cheap', 'Lo', 'Hi']), :]

## Exercise 1
Do a RIGHT OUTER JOIN on the dataset and compare the result to the LEFT OUTER JOIN we've done during the class.

1. How do the `Mercedes Benz` rows compare?
2. How do the "invented rows" from `avg_model_prices` compare?
3. What's the shape of the result? Explain what you think it should be and then check it.
4. How can you achieve the behaviour of the LEFT OUTER JOIN from the class, but doing the RIGHT OUTER JOIN (`merge` with `how='right' parameter`)?

In [ ]:
right_join = pd.merge(cars, avg_model_prices, on=['Make', 'Model'], how='right')

In [ ]:
# 1. How do the Mercedes Benz rows compare?

# The rows of mercedes cars are not included in the result.
# RIGHT OUTER JOIN only keeps matched rows and unmatched rows from the right dataframe.
# avg_model_prices (right) doesn't contain any 'Mercedes Benz' rows --> nothing to include
# LEFT OUTER JOIN (from the class) includes 'Mercedes Benz' rows (from cars) and
# fills NaN values for the avg_model_prices (i.e. avg_price)
filter_mercedes_rows(right_join)

# 2. How do the "invented rows" from avg_model_prices compare?

# The invented rows are included in the result (due to RIGHT OUTER JOIN behaviour explained above).
# Columns from cars are filled in with NaN.
# Contrast that to LEFT OUTER JOIN (from the class) which doesn't include the rows.
filter_invented_rows(right_join)

# 3. What's the shape of the result? Explain what you think it should be and then check it.

# The number of rows should be: numRows(cars) - numRows(mercedes cars) + 3 invented rows = 9747
# The number of columns should be: numColumns(cars) + 1 (avg_price from avg_model_prices). Make and Model are not repeated in the result

right_join.shape

# 4. How can you achieve the behaviour of the LEFT OUTER JOIN from the class, but doing the RIGHT OUTER JOIN (`merge` with `how='right' parameter`)?

# Simply swap the cars and avg_model_prices arguments to the `merge` function

## Exercise 2
Do a FULL OUTER JOIN and INNER JOIN on the datasets and answer following questions

  1. Check the shape of the result and compare it to the shape of cars/avg_model_prices.
    * What do you observe and how can you explain it?
  2. Check how `Mercedes Benz` rows look in the result and explain why
  3. Check how the "invented rows" from `avg_model_prices` look in the result and explain why

In [ ]:
# FULL OUTER JOIN

# 1. Check the shape of the result and compare it to the shape of cars/avg_model_prices.

# The result contains 10,003 rows. This is numRows(cars) + 3 (invented rows from avg_model_prices)
# A FULL OUTER JOIN includes all matched rows and all unmatched rows from both tables in the result.
# Every row from cars has at most one match in avg_model_prices. Therefore, no more
# rows are generated in the result and the 3 additional rows come from the unmatched
# "invented" rows of avg_model_prices.
# Another way to put the result rows:
# numMatchedRows + numUnmatchedRowsLeft (i.e. mercedes cars) + numUnmatchedRowsRight (i.e. invented rows)

outer_join = pd.merge(cars, avg_model_prices, on=['Make', 'Model'], how='outer')
outer_join.shape

# 2. Check how Mercedes Benz rows look in the result and explain why

# All column values filled in from cars and NaN values for avg_price from avg_model_prices,
# because avg_model_prices doesn't contain any mercedes rows.
filter_mercedes_rows(outer_join).head()

# 3. Check how the "invented rows" from `avg_model_prices` look in the result and explain why
# Make, Model and avg_price are filled in. All columns that are only in cars are filled with NaN,
# because cars doesn't contain any of the invented rows' (Make,Model) pair
filter_invented_rows(outer_join)

In [ ]:
# INNER JOIN

# 1. Check the shape of the result and compare it to the shape of cars/avg_model_prices.

# The result contains 9,744 rows. This is numRows(cars) - numRows(mercedes cars)
# An INNER JOIN only includes the matched rows. We know that the mercedes rows from
# cars don't have a match in avg_model_prices and cars doesn't contain any of the
# invented (Make,Model) pairs from avg_model_prices. Additionally, the (Make,Model)
# pairs in avg_model_prices are unique. Therefore, the join produces at most one
# output row for each input row of the cars dataframe.
inner_join = pd.merge(cars, avg_model_prices, on=['Make', 'Model'], how='inner')
inner_join.shape

# 2. Check how Mercedes Benz rows look in the result and explain why

# Empty for the reasons explained above. avg_model_prices doesn't contain mercedes rows
filter_mercedes_rows(inner_join)

# Empty for the reasons explained above. cars doesn't contain invented rows
filter_invented_rows(inner_join)

## Exercise 3
Compute the price difference (based on the euros price) of each car, compared to the average price of each brand (`Make`). Do a join as part of the solution. Which type of join do you use? Explain your choice.

In [ ]:
avg_make_prices = cars.groupby(['Make'])['price_euros'].mean().reset_index()
avg_make_prices = avg_make_prices.rename(columns={'price_euros': 'avg_make_price'})

# The choice of the join in fact doesn't matter: avg_make_prices contains every
# unique Make from cars. Therefore, it is guaranteed that every row from cars matches
# exactly 1 row from avg_make_prices. inner, outer, right, left behave the
# same in this case.
cars_with_avg_makeprice = pd.merge(cars, avg_make_prices, on=['Make'], how='inner')

cars_with_avg_makeprice.loc[:, 'price_diff'] = cars_with_avg_makeprice['price_euros'] - cars_with_avg_makeprice['avg_make_price']

# not needed
cars_with_avg_makeprice.loc[:, 'abs_price_diff'] = cars_with_avg_makeprice.loc[:, 'price_diff'].abs()

# Bonus Dataset: IMDB movie dataset
We'll look at the movie datasets provided by [IMDb](https://www.imdb.com/). The datasets are described [here](https://developer.imdb.com/non-commercial-datasets/). We'll work with a sample of the `title.basics` dataset (which includes only the top 10,000 most-voted movies) and `title.ratings` datasets.

In [ ]:
movies = pd.read_csv("https://raw.githubusercontent.com/obreit/redi/master/imdb_data/top_voted.tsv", sep='\t', header=0)

## Exercise 1
The goal is to get long-running movies (i.e. with `runtimeMinutes` more than 2 hours). Notice that the obvious expression `movies['runtimeMinutes'] > 120` crashes. Try it yourself and try to explain why this is not working (for a hint, see the [details](https://developer.imdb.com/non-commercial-datasets/#imdb-dataset-details) section of the imdb page). Before going to the next hints, try to implement a solution to this problem by yourself.

The `runtimeMinutes` column isn't of numeric type, because it contains `\N` values that can't be interpreted as numbers. So we need to somehow update those values and make a numeric column out of it.
1. Replace all rows where `runtimeMinutes` contains the imdb encoding for a missing value (`\N`) with `pd.NA` (the pandas type which represents a missing/null value). Note that you might run into some syntax issues. Try to find out how to overcome those.
2. Try to cast the column to a numeric type like this: `.astype(int)`. Why doesn't it work?
3. Try to find alternative ways to make the casting work.
4. Filter the movies that are longer than 2 hours

In [ ]:
movies.loc[movies['runtimeMinutes']=='\\N', 'runtimeMinutes'] = pd.NA

# alternatively pd.to_numeric(movies['runtimeMinutes'])
movies['runtimeMinutes'] = movies['runtimeMinutes'].astype(pd.Int64Dtype())

movies[movies['runtimeMinutes'] > 120].head(20)

## Exercise 2
Count how many movies there are per genre. Notice that the `genres` column contains the information about the genre of a movie. But a movie can be assigned to multiple genres. We want to count the movie for every genre. However, the `genres` column is not in a format that makes this counting easy (it's simply a string column). You'll need to transform the `genres` column in a way to simplify the counting. Part of this transformation will be to use the [explode](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.explode.html) function.   

In [ ]:
movies['genre'] = movies['genres'].str.split(',')
movies_flat_genre = movies.explode('genre')

movies_flat_genre['genre'].value_counts()

## Exercise 3
Find out some information about the movie ratings. Specifically, answer following questions
1. What's the movie with the highest average rating?
2. What's the movie with the lowest average rating among the movies with at least 1 million votes (if there are multiple with the same rating, return all of them)
3. Look at the last 10 years and count the number of movies per year as well as the average rating per year

In order to answer the questions, you need to combine the `ratings` dataset with the movies dataset.

In [ ]:
ratings = pd.read_csv("https://datasets.imdbws.com/title.ratings.tsv.gz", sep='\t', header=0)

In [ ]:
movies_with_ratings = movies.merge(ratings, on='tconst')

In [ ]:
# 1.
movies_with_ratings.loc[movies_with_ratings['averageRating'].idxmax()]

In [ ]:
# 2.
movies_with_many_votes = movies_with_ratings[movies_with_ratings['numVotes'] > 1000000]

lowest_rating = movies_with_many_votes['averageRating'].min()

movies_with_many_votes[movies_with_many_votes['averageRating']==lowest_rating]

In [ ]:
# 3.
movies_last_decade = movies_with_ratings[movies_with_ratings['startYear'] >= 2013]

movies_last_decade.groupby('startYear').agg(count=('tconst', np.size), averageRating=('averageRating', np.mean))